# Experience Memory Experiments

This notebook runs the EOH memory conditions one experiment at a time.

Recommended order:
1. `A_vanilla_eoh`
2. `B_memory_write_only`
3. `C_memory_retrieval`
4. `D_memory_retrieval_plus_seed`

`C` and `D` should reuse the memory file produced by `B`.

In [ ]:
from pathlib import Path
import json
import os
import sys

from hpc_llm_setup import ensure_eoh_src_on_path

project_root, eoh_src = ensure_eoh_src_on_path(Path.cwd())
notebooks_dir = project_root / 'notebooks'
if str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

project_root, eoh_src

In [ ]:
# Shared configuration used by all experiment cells
os.environ['EOH_MEMORY_COMPARE_OUT'] = str(project_root / 'compare_runs' / 'experience_memory')
os.environ['EOH_MEMORY_STORE_PATH'] = str(project_root / 'compare_runs' / 'experience_memory' / 'shared_memory' / 'experience_memory.jsonl')
os.environ['EOH_SHARE_INITIAL_POP'] = '1'
os.environ['EOH_POP_SIZE'] = '8'
os.environ['EOH_N_GENERATIONS'] = '10'
os.environ['EOH_N_PROC'] = '32'
os.environ['EOH_EVAL_PARALLEL_INSTANCES'] = '32'
os.environ['EOH_EVAL_INSTANCES_PER_GEN'] = '256'
os.environ['EOH_HOLDOUT_INSTANCES'] = '64'
os.environ['EOH_MEMORY_TOP_K'] = '3'
os.environ['EOH_MEMORY_SEED_TOP_N'] = '2'

# Keep runtime behavior aligned with the current HPC bridge defaults
os.environ.setdefault('EOH_PARALLEL_BACKEND', 'threading')
os.environ.setdefault('EOH_PARALLEL_FALLBACK_SEQUENTIAL', '1')
os.environ.setdefault('EOH_OFFSPRING_RETRIES', '4')
os.environ.setdefault('EOH_PARSE_RETRIES', '2')
os.environ.setdefault('EOH_MAX_PARALLEL_LLM_REQUESTS', '1')
os.environ.setdefault('EOH_MAX_PARALLEL_I1_REQUESTS', '1')
os.environ.setdefault('EOH_GENERATION_TIMEOUT_S', '900')
os.environ.setdefault('EOH_LOCAL_LLM_TIMEOUT_S', '300')
os.environ.setdefault('EOH_LOCAL_LLM_MAX_RETRIES', '1')
os.environ.setdefault('EOH_LOCAL_LLM_MAX_NEW_TOKENS_CODE', '512')
os.environ.setdefault('EOH_LOCAL_LLM_MAX_NEW_TOKENS_JSON', '1400')

{k: os.environ[k] for k in [
    'EOH_MEMORY_COMPARE_OUT',
    'EOH_MEMORY_STORE_PATH',
    'EOH_SHARE_INITIAL_POP',
    'EOH_POP_SIZE',
    'EOH_N_GENERATIONS',
    'EOH_N_PROC',
    'EOH_EVAL_PARALLEL_INSTANCES',
    'EOH_EVAL_INSTANCES_PER_GEN',
    'EOH_HOLDOUT_INSTANCES',
    'EOH_MEMORY_TOP_K',
    'EOH_MEMORY_SEED_TOP_N',
]}

## Run A: Vanilla EOH

In [ ]:
os.environ['EOH_MEMORY_EXPERIMENTS'] = 'A_vanilla_eoh'
os.environ['EOH_MEMORY_RESET_STORE'] = '1'
%run run_memory_experiments.py

## Run B: Memory Write Only

This should populate the shared memory store for later retrieval experiments.

In [ ]:
os.environ['EOH_MEMORY_EXPERIMENTS'] = 'B_memory_write_only'
os.environ['EOH_MEMORY_RESET_STORE'] = '1'
%run run_memory_experiments.py

## Run C: Memory Retrieval

Use this after `B_memory_write_only` has already created the shared memory file.

In [ ]:
os.environ['EOH_MEMORY_EXPERIMENTS'] = 'C_memory_retrieval'
os.environ['EOH_MEMORY_RESET_STORE'] = '0'
%run run_memory_experiments.py

## Run D: Memory Retrieval Plus Seed

Use this after `B_memory_write_only` has already created the shared memory file.

In [ ]:
os.environ['EOH_MEMORY_EXPERIMENTS'] = 'D_memory_retrieval_plus_seed'
os.environ['EOH_MEMORY_RESET_STORE'] = '0'
%run run_memory_experiments.py

## Inspect Summary

In [ ]:
summary_path = Path(os.environ['EOH_MEMORY_COMPARE_OUT']) / 'memory_experiment_summary.json'
with summary_path.open('r', encoding='utf-8') as f:
    summary = json.load(f)
summary

In [ ]:
for item in summary.get('experiments', []):
    mem = item.get('memory_usage', {})
    print(
        Path(item.get('experiment_root', '')).name,
        'final_best_fitness=', item.get('final_best_fitness'),
        'memory_entries_written=', mem.get('memory_entries_written'),
        'memory_retrieval_events=', mem.get('memory_retrieval_events'),
        'memory_seeded_ids=', len(mem.get('memory_seeded_ids', [])),
    )